# 05 — Danh Gia Tong Hop & Chuoi Thoi Gian
Drift Detection - Time-series Split - Ket luan

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.data.loader         import load_config
from src.features.builder    import FeatureBuilder
from src.models.supervised   import Trainer
from src.models.forecasting  import TimeSeriesAnalyzer
from src.evaluation.report   import Reporter
from src.visualization.plots import plot_trend, plot_drift

cfg      = load_config("../configs/params.yaml")
df_clean = pd.read_parquet("../data/processed/yield_clean.parquet")
reporter = Reporter(cfg)
print("Setup OK")

## 5.1 Trend nang suat theo nam

In [ ]:
fb = FeatureBuilder(cfg)
fb.fit_transform(df_clean)

ts = TimeSeriesAnalyzer(cfg)
year_mean = ts.analyze_trend(df_clean)
print(f"Trend: {ts.trend_coef_:+.1f} hg/ha/nam")

plot_trend(year_mean, ts.trend_poly_, ts.trend_coef_, cfg["paths"]["figures"])
plt.show()

## 5.2 Drift Detection (KS-Test)

In [ ]:
drift = ts.drift_detection(df_clean)
for k, v in drift.items():
    print(f"  {k}: {v}")

plot_drift(ts.y_early_, ts.y_late_,
           drift["years_early"], drift["years_late"],
           drift["ks_pval"], cfg["paths"]["figures"])
plt.show()

## 5.3 He so bien dong (CV%)

In [ ]:
cv_df = ts.cv_analysis(df_clean)
fig, ax = plt.subplots(figsize=(11,3))
ax.bar(cv_df["Year"], cv_df["CV_pct"], color="mediumorchid",
       edgecolor="white", alpha=0.8)
ax.axhline(cv_df["CV_pct"].mean(), color="red", linestyle="--",
           label=f"TB={cv_df['CV_pct'].mean():.1f}%")
ax.set_title("He so bien dong (CV%) Yield theo nam", fontweight="bold")
ax.set_xlabel("Nam"); ax.set_ylabel("CV (%)")
ax.legend()
plt.tight_layout()
plt.show()

## 5.4 Time-Series Split vs Random Split

In [ ]:
trainer = Trainer(cfg)
trainer.fit_all(fb.X_train, fb.X_test, fb.y_train, fb.y_test,
                fb.X_train_s, fb.X_test_s)

ts_result = ts.time_split_train(fb)
rf_mae = trainer.results_.get("RandomForest", {}).get("MAE", 0)
reporter.print_time_series(ts_result, rf_mae)
reporter.save_table(pd.DataFrame([ts_result]), "timeseries_split.csv")
reporter.save_table(pd.DataFrame([drift]),     "drift_detection.csv")

## 5.5 Ket luan tong hop

In [ ]:
best_row = trainer.df_results_.iloc[0]
reporter.print_summary(
    cfg,
    n_raw=len(df_clean),
    n_clean=len(df_clean),
    best_model=best_row["Model"],
    best_mae=best_row["MAE"],
    best_r2=best_row["R2"],
    n_rules=0,
    k_clusters=cfg["clustering"]["k_best"]
)

In [ ]:
insights = [
    "Loai cay va vung dia ly la 2 yeu to quyet dinh nhat (feature importance cao nhat).",
    "Luong mua 500-1500mm + nhiet do 15-25C -> Yield cao theo Association Rules.",
    "Cac cum vung trong tach biet ro theo dieu kien khi hau (Silhouette > 0.4).",
    "RF/XGBoost vuot troi mo hinh tuyen tinh -> quan he phi tuyen chiem uu the.",
    "Vung Yield hiem (>95%ile): loi tuong doi tang 2-3x -> can SMOTE/augmentation.",
    "Trend tang theo nam -> tien bo ky thuat canh tac phan anh trong du lieu.",
]
print("INSIGHTS VA KHUYEN NGHI:")
for i, ins in enumerate(insights, 1):
    print(f"  {i}. {ins}")